In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()

print("Setup complete")

Setup complete


In [3]:
import gradio as gr

def summarize_pros_cons(topic, num_points):
    prompt = f"List {num_points} pros and cons of: {topic}"
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

demo = gr.Interface(
    fn=summarize_pros_cons,
    inputs=["text", gr.Slider(1, 5, step=1)],
    outputs="text"
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [2]:
import gradio as gr

def summarize_pros_cons(topic, num_points):
    prompt = f"List {num_points} pros and cons of: {topic}"
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

demo = gr.Interface(
    fn=summarize_pros_cons,
    inputs=["text", gr.Slider(1, 5, value=3, step=1)],
    outputs="text"
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [4]:
def chat_stream(message):
    stream = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": message}],
        stream=True
    )

    partial_reply = ""
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            partial_reply += content
            yield partial_reply

demo = gr.Interface(fn=chat_stream, inputs="text", outputs="text")
demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [5]:
import gradio as gr
from anthropic import Anthropic

anthropic_client = Anthropic()

def stream_gpt(message):
    stream = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": message}],
        stream=True
    )
    reply = ""
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            reply += content
            yield reply

def stream_claude(message):
    reply = ""
    with anthropic_client.messages.stream(
        model="claude-sonnet-4-5",
        max_tokens=500,
        messages=[{"role": "user", "content": message}]
    ) as stream:
        for text in stream.text_stream:
            reply += text
            yield reply

with gr.Blocks() as demo:
    prompt = gr.Textbox(label="Your question")
    with gr.Row():
        gpt_output = gr.Textbox(label="GPT-5")
        claude_output = gr.Textbox(label="Claude")
    submit = gr.Button("Ask both")
    submit.click(stream_gpt, inputs=prompt, outputs=gpt_output)
    submit.click(stream_claude, inputs=prompt, outputs=claude_output)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Chat UI with real conversation memory.

In [9]:
import gradio as gr
print(gr.__version__)

6.27.0


In [10]:
def chat(message, history):
    messages = history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=messages
    )
    return response.choices[0].message.content

demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [11]:
def chat(message, history):
    system_message = {"role": "system", "content": "You are a sarcastic assistant who answers correctly but with dry wit."}
    messages = [system_message] + history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=messages
    )
    return response.choices[0].message.content

demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [12]:
def chat(message, history):
    system_message = {"role": "system", "content": "You convert casual sentences into formal business English."}

    examples = [
        {"role": "user", "content": "hey can u send me that file"},
        {"role": "assistant", "content": "Could you please send me that file at your earliest convenience?"},
        {"role": "user", "content": "gonna be late sry"},
        {"role": "assistant", "content": "I apologize, but I will be arriving later than scheduled."}
    ]

    messages = [system_message] + examples + history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
    return response.choices[0].message.content

demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [13]:
def chat(message, history):
    system_message = {"role": "system", "content": "You convert casual sentences into formal business English."}

    examples = [
        {"role": "user", "content": "hey can u send me that file"},
        {"role": "assistant", "content": "Could you please send me that file at your earliest convenience?"},
        {"role": "user", "content": "gonna be late sry"},
        {"role": "assistant", "content": "I apologize, but I will be arriving later than scheduled."}
    ]

    messages = [system_message] + examples + history + [{"role": "user", "content": message}]

    response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
    return response.choices[0].message.content

demo = gr.ChatInterface(fn=chat)
demo.launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [19]:
import json

def get_ticket_price(destination_city):
    prices = {"london": "£299", "paris": "£210", "tokyo": "£850"}
    city = destination_city.lower()
    return prices.get(city, "Unknown destination")


#get_ticket_price("london")


price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city. Call this whenever a user asks about ticket prices, for example 'How much is a ticket to Paris?'",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city the customer wants to travel to"
            }
        },
        "required": ["destination_city"]
    }
}

tools = [{"type": "function", "function": price_function}]

##pass tools into the API call and see what happens when the model decides to use one:
"""
messages = [{"role": "user", "content": "How much is a ticket to Tokyo?"}]

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=messages,
    tools=tools
)

print(response.choices[0].finish_reason)
print(response.choices[0].message.tool_calls)

"""



tool_call = response.choices[0].message.tool_calls[0]
arguments = json.loads(tool_call.function.arguments)
city = arguments["destination_city"]

price = get_ticket_price(city)

messages.append(response.choices[0].message)
messages.append({
    "role": "tool",
    "content": json.dumps({"destination_city": city, "price": price}),
    "tool_call_id": tool_call.id
})

final_response = client.chat.completions.create(model="gpt-5-mini", messages=messages)
print(final_response.choices[0].message.content)


A ticket to Tokyo is £850 (GBP). 

Would you like me to check specific dates, one-way vs round-trip, cabin class (economy/business), or convert that to another currency?
